# 01 · Stage 1: row counts

Left counts per partition come from the source; Right counts are one `GROUP BY` on the warehouse
(only the counts travel). Both are reconciled here: `match`, `count_diff`, `left_only`, `right_only`.
The gate decides which pairs move on; only fully matching partitions are compared in Stage 2.

Re-running is safe: a pair that already has its verdict (`row_check/<pair>.json`) is not recounted
on the Right side. To start over, use a new `RUN_ID`.

In [ ]:
# Parameters: keep CONFIG / OUT identical in notebooks 00-03
CONFIG = "c.json"  # pairs config (col_map files resolve next to it)
OUT = ".dtrack-local/runs"  # local runs root; one folder per run id
RUN_ID = None  # None: 00 starts a new run, 01-03 continue the run 00 opened (OUT/LATEST)
DISPOSITION = (
    "external_links"  # "inline" when presigned downloads are blocked (<= 25 MiB per window)
)
PUSH_SAMPLE = True  # False only when you need to land the full Right window locally
PAIRS = None  # e.g. ["orders"]; None = every active pair
GATE = "auto"  # auto | manual | ratio:<max excluded share>

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 40, "display.width", 200)

from dtrack_local import pipeline

In [ ]:
RUN_ID = RUN_ID or (Path(OUT) / "LATEST").read_text().strip()
run = pipeline.open_run(
    CONFIG, run_id=RUN_ID, out=OUT, disposition=DISPOSITION, push_sample=PUSH_SAMPLE, pairs=PAIRS
)
print("run_id:", run.run_id)
print("folder:", run.folder.resolve())
print("status:", run.status())

## Count and reconcile

In [ ]:
stage_one = pipeline.row_check(run, gate=GATE)
pd.DataFrame(
    [
        {"pair": pair, **(row["summary"] or {}), "gate": (row["gate"] or {}).get("passed")}
        for pair, row in stage_one.items()
    ]
)

## Partitions that did not match

In [ ]:
mismatches = []
for pair in stage_one:
    document = run.store.get_json(f"row_check/{pair}.json")
    for partition, row in document["partitions"].items():
        if row["verdict"] != "match":
            mismatches.append({"pair": pair, "partition": partition, **row})
pd.DataFrame(mismatches) if mismatches else "every partition matched"

## What Stage 2 will do
One unit = one pair x vintage window x column.

In [ ]:
plans = []
for key, manifest in run.store.list_json("manifest/"):
    if key == "manifest/_CLOSED" or key.endswith(".keys.json"):
        continue
    plans.append(
        {
            "pair": manifest["pair"],
            "windows": len(manifest["windows"]),
            "columns": len(manifest["columns"]),
            "units": manifest["expected_units"],
            "sample": manifest["sample"]["mode"],
            "rows_left": sum(window["n_left"] for window in manifest["windows"]),
        }
    )
print("row_check/_CLOSED:", run.store.exists("row_check/_CLOSED"))
pd.DataFrame(plans)